In [ ]:
# =============================================================================
# FRAUD DETECTION USING AUTOENCODERS
# Complete Google Colab Training Pipeline
# =============================================================================
#
# Dataset:
# Kaggle - MLG-ULB Credit Card Fraud Detection
#
# Project methodology:
# 1. Load and validate dataset
# 2. Perform essential EDA
# 3. Separate features and fraud labels
# 4. Create a held-out test set
# 5. Use ONLY normal transactions for model development
# 6. Standardize using scaler fitted ONLY on normal training data
# 7. Train Autoencoder using normal transactions only
# 8. Calculate reconstruction error
# 9. Select threshold using NORMAL validation reconstruction errors
# 10. Evaluate on completely held-out test data
# 11. Compare with Isolation Forest
# 12. Compare with One-Class SVM
# 13. Calculate Precision, Recall, F1-score and fraud cases detected
# 14. Save trained models, scaler, thresholds and results
#
# IMPORTANT:
# The "Class" fraud label is NOT used to train the Autoencoder,
# Isolation Forest, or One-Class SVM.
# Class labels are used ONLY for final evaluation.
# =============================================================================


# =============================================================================
# INSTALL LIBRARIES
# =============================================================================

!pip -q install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib


# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import os
import random
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from IPython.display import display

from tensorflow.keras import Model, Input
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)

warnings.filterwarnings("ignore")

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 80)
print("FRAUD DETECTION USING AUTOENCODERS")
print("=" * 80)

print("\nTensorFlow version:", tf.__version__)
print("Random seed:", SEED)


# =============================================================================
# LOAD DATASET — UPLOADING
# =============================================================================

from google.colab import files
import io
import pandas as pd

print("Please select the CSV file from your computer.")

uploaded = files.upload()

if not uploaded:
    raise FileNotFoundError(
        "No file was uploaded. Please upload CSV File."
    )

# Get the uploaded filename
file_name = list(uploaded.keys())[0]

# Verify CSV file
if not file_name.lower().endswith(".csv"):
    raise ValueError(
        "Please upload a CSV file. Expected: creditcard.csv"
    )

# Load CSV into DataFrame
df = pd.read_csv(
    io.BytesIO(uploaded[file_name])
)

print("\nDataset loaded successfully.")
print("Uploaded file:", file_name)
print("Dataset shape:", df.shape)


# =============================================================================
# DATA VALIDATION
# =============================================================================

print("\n" + "=" * 80)
print("DATA VALIDATION")
print("=" * 80)

print("\nFirst 5 rows:")
display(df.head())

print("\nDataset information:")
df.info()

print("\nMissing values:")
total_missing = df.isnull().sum().sum()
print("Total missing values:", total_missing)

if total_missing == 0:
    print("✓ No missing values found.")

print("\nDuplicate rows:")
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes)


# =============================================================================
# BASIC DATASET CHECKS
# =============================================================================

required_columns = (
    ["Time"]
    + [f"V{i}" for i in range(1, 29)]
    + ["Amount", "Class"]
)

missing_required_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_required_columns:
    raise ValueError(
        f"Required columns are missing: {missing_required_columns}"
    )

print("\n✓ Required dataset structure verified.")

print("\nClass values:")
print(sorted(df["Class"].unique()))

if not set(df["Class"].unique()).issubset({0, 1}):
    raise ValueError("Class column must contain only 0 and 1.")

print("✓ Class column verified.")


# =============================================================================
# CLASS DISTRIBUTION
# =============================================================================

print("\n" + "=" * 80)
print("CLASS DISTRIBUTION")
print("=" * 80)

class_counts = df["Class"].value_counts().sort_index()

print("\nTransaction counts:")
print(class_counts)

class_percentages = (
    df["Class"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(4)
)

print("\nTransaction percentages:")
print(class_percentages)

normal_count = int((df["Class"] == 0).sum())
fraud_count = int((df["Class"] == 1).sum())

print("\nNormal transactions:", f"{normal_count:,}")
print("Fraud transactions:", f"{fraud_count:,}")
print("Fraud percentage:", f"{fraud_count / len(df) * 100:.4f}%")


# =============================================================================
# EDA PLOT 1: CLASS DISTRIBUTION
# =============================================================================

plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=df,
    x="Class"
)

plt.title("Normal vs Fraudulent Transactions")
plt.xlabel("Class (0 = Normal, 1 = Fraud)")
plt.ylabel("Number of Transactions")

for container in ax.containers:
    ax.bar_label(container)

plt.tight_layout()
plt.show()


# =============================================================================
# EDA PLOT 2: TRANSACTION AMOUNT
# =============================================================================

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Amount",
    hue="Class",
    bins=100,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Transaction Amount Distribution")
plt.xlabel("Transaction Amount")
plt.ylabel("Density")

plt.tight_layout()
plt.show()


# =============================================================================
# EDA PLOT 3: TIME DISTRIBUTION
# =============================================================================

plt.figure(figsize=(10, 5))

sns.histplot(
    data=df,
    x="Time",
    hue="Class",
    bins=100,
    element="step",
    stat="density",
    common_norm=False
)

plt.title("Transaction Time Distribution")
plt.xlabel("Time")
plt.ylabel("Density")

plt.tight_layout()
plt.show()


# =============================================================================
# AMOUNT STATISTICS
# =============================================================================

print("\nTransaction amount statistics by class:")

amount_statistics = (
    df.groupby("Class")["Amount"]
    .describe()
    .round(2)
)

display(amount_statistics)


# =============================================================================
# DEFINE FEATURES AND LABEL
# =============================================================================

TARGET = "Class"

FEATURES = [
    col for col in df.columns
    if col != TARGET
]

X = df[FEATURES].copy()
y = df[TARGET].copy()

print("\n" + "=" * 80)
print("FEATURE / LABEL SEPARATION")
print("=" * 80)

print("\nNumber of input features:", len(FEATURES))
print("Feature shape:", X.shape)
print("Label shape:", y.shape)

print("\nFeatures:")
print(FEATURES)

print("\nIMPORTANT:")
print("Class label will NOT be used as a model input.")


# =============================================================================
# HELD-OUT TEST SPLIT
# =============================================================================
#
# We reserve 20% of the complete dataset as a final held-out test set.
# The test labels remain untouched until final evaluation.
#
# stratify=y keeps the rare fraud proportion represented in the test set.
# This is ONLY a data split operation and does not train a model.
# =============================================================================

X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
    stratify=y
)

print("\n" + "=" * 80)
print("HELD-OUT TEST SET")
print("=" * 80)

print("\nDevelopment data:", X_dev.shape)
print("Held-out test data:", X_test.shape)

print("\nFraud cases in development data:", int(y_dev.sum()))
print("Fraud cases in held-out test data:", int(y_test.sum()))


# =============================================================================
# NORMAL-ONLY DEVELOPMENT DATA
# =============================================================================
#
# CRITICAL PROJECT REQUIREMENT:
# Only Class = 0 transactions are allowed for model training/development.
#
# Fraud labels are used here ONLY to filter normal transactions.
# No fraud transaction is passed into model fitting.
# =============================================================================

normal_mask = y_dev == 0

X_normal = X_dev.loc[normal_mask].copy()

print("\n" + "=" * 80)
print("NORMAL-ONLY MODEL DEVELOPMENT DATA")
print("=" * 80)

print("\nNormal transactions available:", X_normal.shape[0])
print("Fraud transactions excluded:", int((y_dev == 1).sum()))

print("\n✓ Fraud transactions are excluded from Autoencoder training.")


# =============================================================================
# NORMAL TRAIN / VALIDATION SPLIT
# =============================================================================

X_train_normal, X_val_normal = train_test_split(
    X_normal,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

print("\nNormal training samples:", X_train_normal.shape[0])
print("Normal validation samples:", X_val_normal.shape[0])


# =============================================================================
# STANDARD SCALER
# =============================================================================
#
# IMPORTANT:
# The scaler is fitted ONLY on normal training data.
#
# It is then applied to:
# - normal validation data
# - held-out test data
#
# This scaler will later be saved and used by Streamlit.
# =============================================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = scaler.transform(X_val_normal)
X_test_scaled = scaler.transform(X_test)

print("\n" + "=" * 80)
print("FEATURE SCALING")
print("=" * 80)

print("\nTraining shape:", X_train_scaled.shape)
print("Validation shape:", X_val_scaled.shape)
print("Test shape:", X_test_scaled.shape)

print("\n✓ Scaler fitted ONLY on normal training data.")


# =============================================================================
# BUILD AUTOENCODER
# =============================================================================

INPUT_DIM = X_train_scaled.shape[1]

input_layer = Input(
    shape=(INPUT_DIM,),
    name="transaction_input"
)

# ------------------------- Encoder -------------------------

encoded = Dense(
    128,
    activation="relu",
    name="encoder_128"
)(input_layer)

encoded = Dense(
    64,
    activation="relu",
    name="encoder_64"
)(encoded)

encoded = Dense(
    32,
    activation="relu",
    name="encoder_32"
)(encoded)

bottleneck = Dense(
    16,
    activation="relu",
    name="bottleneck"
)(encoded)

# ------------------------- Decoder -------------------------

decoded = Dense(
    32,
    activation="relu",
    name="decoder_32"
)(bottleneck)

decoded = Dense(
    64,
    activation="relu",
    name="decoder_64"
)(decoded)

decoded = Dense(
    128,
    activation="relu",
    name="decoder_128"
)(decoded)

output_layer = Dense(
    INPUT_DIM,
    activation="linear",
    name="reconstruction"
)(decoded)

autoencoder = Model(
    inputs=input_layer,
    outputs=output_layer,
    name="CreditCardFraudAutoencoder"
)

autoencoder.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse"
)

print("\n" + "=" * 80)
print("AUTOENCODER ARCHITECTURE")
print("=" * 80)

autoencoder.summary()


# =============================================================================
# TRAIN AUTOENCODER
# =============================================================================
#
# Input  = normal transaction
# Target = same normal transaction
#
# No fraud transactions are used.
# =============================================================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

print("\n" + "=" * 80)
print("TRAINING AUTOENCODER")
print("=" * 80)

history = autoencoder.fit(
    X_train_scaled,
    X_train_scaled,
    validation_data=(
        X_val_scaled,
        X_val_scaled
    ),
    epochs=100,
    batch_size=256,
    shuffle=True,
    callbacks=[
        early_stopping,
        reduce_lr
    ],
    verbose=1
)

print("\n✓ Autoencoder training completed.")
print("Best weights restored using validation loss.")


# =============================================================================
# TRAINING LOSS PLOT
# =============================================================================

plt.figure(figsize=(10, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.title("Autoencoder Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.legend()

plt.tight_layout()
plt.show()


# =============================================================================
# RECONSTRUCTION ERROR FUNCTION
# =============================================================================

def calculate_reconstruction_error(model, data):

    reconstructed = model.predict(
        data,
        verbose=0
    )

    errors = np.mean(
        np.square(
            data - reconstructed
        ),
        axis=1
    )

    return errors


# =============================================================================
# NORMAL VALIDATION RECONSTRUCTION ERROR
# =============================================================================

val_errors = calculate_reconstruction_error(
    autoencoder,
    X_val_scaled
)

print("\n" + "=" * 80)
print("VALIDATION RECONSTRUCTION ERROR")
print("=" * 80)

print("\nMinimum:", f"{val_errors.min():.8f}")
print("Maximum:", f"{val_errors.max():.8f}")
print("Mean:", f"{val_errors.mean():.8f}")
print("Median:", f"{np.median(val_errors):.8f}")


# =============================================================================
# RECONSTRUCTION ERROR DISTRIBUTION
# =============================================================================

plt.figure(figsize=(10, 5))

plt.hist(
    val_errors,
    bins=100
)

plt.title(
    "Reconstruction Error Distribution - Normal Validation Data"
)

plt.xlabel("Reconstruction Error")
plt.ylabel("Number of Transactions")

plt.tight_layout()
plt.show()


# =============================================================================
# THRESHOLD ANALYSIS
# =============================================================================
#
# IMPORTANT:
# Fraud labels are NOT used for threshold selection.
#
# Thresholds are calculated ONLY from normal validation reconstruction errors.
#
# We evaluate:
# 95th percentile
# 97.5th percentile
# 99th percentile
# 99.5th percentile
# =============================================================================

percentiles = [
    95,
    97.5,
    99,
    99.5
]

threshold_results = []

for percentile in percentiles:

    threshold = np.percentile(
        val_errors,
        percentile
    )

    normal_flagged = int(
        np.sum(val_errors > threshold)
    )

    normal_total = len(val_errors)

    false_alarm_rate = (
        normal_flagged /
        normal_total
    )

    threshold_results.append({
        "Percentile": percentile,
        "Threshold": threshold,
        "Normal Transactions Flagged": normal_flagged,
        "Normal False Alarm Rate": false_alarm_rate
    })

threshold_df = pd.DataFrame(
    threshold_results
)

print("\n" + "=" * 80)
print("THRESHOLD ANALYSIS")
print("=" * 80)

display(
    threshold_df.style.format({
        "Threshold": "{:.8f}",
        "Normal False Alarm Rate": "{:.4%}"
    })
)


# =============================================================================
# SELECT FINAL AUTOENCODER THRESHOLD
# =============================================================================
#
# We use the 99th percentile of normal validation reconstruction error.
#
# This means the threshold is determined without looking at fraud labels.
# Approximately the highest 1% of normal validation reconstruction errors
# are treated as potential anomalies.
# =============================================================================

THRESHOLD_PERCENTILE = 99

THRESHOLD = np.percentile(
    val_errors,
    THRESHOLD_PERCENTILE
)

print("\n" + "=" * 80)
print("FINAL AUTOENCODER THRESHOLD")
print("=" * 80)

print(
    f"\nSelected percentile: {THRESHOLD_PERCENTILE}%"
)

print(
    f"Selected threshold: {THRESHOLD:.8f}"
)

print(
    "\nThreshold principle:"
)

print(
    "Transactions with reconstruction error greater than "
    "the threshold are classified as anomalies."
)


# =============================================================================
# THRESHOLD VISUALIZATION
# =============================================================================

plt.figure(figsize=(10, 5))

plt.hist(
    val_errors,
    bins=100
)

plt.axvline(
    THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"99th Percentile Threshold = "
        f"{THRESHOLD:.6f}"
    )
)

plt.title(
    "Normal Validation Reconstruction Errors "
    "with Anomaly Threshold"
)

plt.xlabel("Reconstruction Error")
plt.ylabel("Number of Transactions")

plt.legend()

plt.tight_layout()
plt.show()


# =============================================================================
# HELD-OUT TEST RECONSTRUCTION ERROR
# =============================================================================
#
# This is the first point at which the final test set is evaluated.
#
# y_test remains untouched until predictions are generated.
# =============================================================================

test_errors = calculate_reconstruction_error(
    autoencoder,
    X_test_scaled
)

print("\n" + "=" * 80)
print("HELD-OUT TEST RECONSTRUCTION ERROR")
print("=" * 80)

print("\nMinimum:", f"{test_errors.min():.8f}")
print("Maximum:", f"{test_errors.max():.8f}")
print("Mean:", f"{test_errors.mean():.8f}")
print("Median:", f"{np.median(test_errors):.8f}")


# =============================================================================
# AUTOENCODER PREDICTIONS
# =============================================================================

ae_predictions = (
    test_errors > THRESHOLD
).astype(int)

print("\n" + "=" * 80)
print("AUTOENCODER PREDICTIONS")
print("=" * 80)

print("\nTransactions in test set:", len(y_test))
print("Transactions flagged as anomalies:", int(ae_predictions.sum()))
print("Actual fraud cases:", int(y_test.sum()))


# =============================================================================
# EVALUATION FUNCTION
# =============================================================================

def evaluate_model(
    model_name,
    y_true,
    y_pred
):

    precision = precision_score(
        y_true,
        y_pred,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        zero_division=0
    )

    fraud_detected = int(
        (
            (y_true == 1) &
            (y_pred == 1)
        ).sum()
    )

    total_fraud = int(
        (y_true == 1).sum()
    )

    anomalies_flagged = int(
        (y_pred == 1).sum()
    )

    return {
        "Model": model_name,
        "Precision": precision,
        "Recall": recall,
        "F1-Score": f1,
        "Fraud Cases Detected": fraud_detected,
        "Total Fraud Cases": total_fraud,
        "Transactions Flagged": anomalies_flagged
    }


# =============================================================================
# AUTOENCODER EVALUATION
# =============================================================================

ae_result = evaluate_model(
    "Autoencoder",
    y_test,
    ae_predictions
)

ae_result_df = pd.DataFrame(
    [ae_result]
)

print("\n" + "=" * 80)
print("AUTOENCODER PERFORMANCE")
print("=" * 80)

display(
    ae_result_df.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)


# =============================================================================
# AUTOENCODER CLASSIFICATION REPORT
# =============================================================================

print("\nAUTOENCODER CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        ae_predictions,
        target_names=[
            "Normal",
            "Fraud"
        ],
        zero_division=0
    )
)


# =============================================================================
# AUTOENCODER CONFUSION MATRIX
# =============================================================================

ae_cm = confusion_matrix(
    y_test,
    ae_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    ae_cm,
    annot=True,
    fmt="d",
    square=True
)

plt.title("Autoencoder Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()


# =============================================================================
# AUTOENCODER ROC-AUC
# =============================================================================
#
# Reconstruction error is used as the continuous anomaly score.
# Higher reconstruction error = more anomalous.
# =============================================================================

ae_auc = roc_auc_score(
    y_test,
    test_errors
)

print(
    f"Autoencoder ROC-AUC: {ae_auc:.4f}"
)


# =============================================================================
# ISOLATION FOREST
# =============================================================================
#
# Isolation Forest is trained ONLY on normal training transactions.
# =============================================================================

print("\n" + "=" * 80)
print("ISOLATION FOREST")
print("=" * 80)

isolation_forest = IsolationForest(
    n_estimators=200,
    max_samples="auto",
    contamination="auto",
    random_state=SEED,
    n_jobs=-1
)

isolation_forest.fit(
    X_train_scaled
)

print(
    "\n✓ Isolation Forest trained "
    "using normal training transactions only."
)


# =============================================================================
# ISOLATION FOREST VALIDATION SCORE
# =============================================================================
#
# score_samples:
# Higher = more normal
#
# We multiply by -1:
# Higher anomaly score = more anomalous
# =============================================================================

if_val_scores = -isolation_forest.score_samples(
    X_val_scaled
)

IF_THRESHOLD = np.percentile(
    if_val_scores,
    THRESHOLD_PERCENTILE
)

print(
    "\nIsolation Forest validation threshold:",
    f"{IF_THRESHOLD:.8f}"
)


# =============================================================================
# ISOLATION FOREST TEST PREDICTIONS
# =============================================================================

if_test_scores = -isolation_forest.score_samples(
    X_test_scaled
)

if_predictions = (
    if_test_scores > IF_THRESHOLD
).astype(int)

print(
    "\nTransactions flagged by Isolation Forest:",
    int(if_predictions.sum())
)


# =============================================================================
# ISOLATION FOREST EVALUATION
# =============================================================================

if_result = evaluate_model(
    "Isolation Forest",
    y_test,
    if_predictions
)

print("\nIsolation Forest performance:")

display(
    pd.DataFrame(
        [if_result]
    ).style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)


# =============================================================================
# ONE-CLASS SVM
# =============================================================================
#
# One-Class SVM is also trained ONLY on normal transactions.
# =============================================================================

print("\n" + "=" * 80)
print("ONE-CLASS SVM")
print("=" * 80)

one_class_svm = OneClassSVM(
    kernel="rbf",
    gamma="scale",
    nu=0.01
)

one_class_svm.fit(
    X_train_scaled
)

print(
    "\n✓ One-Class SVM trained "
    "using normal training transactions only."
)


# =============================================================================
# ONE-CLASS SVM VALIDATION SCORE
# =============================================================================
#
# decision_function:
# Higher = more normal
#
# Reverse the sign to obtain:
# Higher = more anomalous
# =============================================================================

ocsvm_val_scores = -one_class_svm.decision_function(
    X_val_scaled
)

OCSVM_THRESHOLD = np.percentile(
    ocsvm_val_scores,
    THRESHOLD_PERCENTILE
)

print(
    "\nOne-Class SVM validation threshold:",
    f"{OCSVM_THRESHOLD:.8f}"
)


# =============================================================================
# ONE-CLASS SVM TEST PREDICTIONS
# =============================================================================

ocsvm_test_scores = -one_class_svm.decision_function(
    X_test_scaled
)

ocsvm_predictions = (
    ocsvm_test_scores > OCSVM_THRESHOLD
).astype(int)

print(
    "\nTransactions flagged by One-Class SVM:",
    int(ocsvm_predictions.sum())
)


# =============================================================================
# ONE-CLASS SVM EVALUATION
# =============================================================================

ocsvm_result = evaluate_model(
    "One-Class SVM",
    y_test,
    ocsvm_predictions
)

print("\nOne-Class SVM performance:")

display(
    pd.DataFrame(
        [ocsvm_result]
    ).style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)


# =============================================================================
# FINAL MODEL COMPARISON
# =============================================================================

comparison_df = pd.DataFrame([
    ae_result,
    if_result,
    ocsvm_result
])

comparison_display = comparison_df[
    [
        "Model",
        "Precision",
        "Recall",
        "F1-Score",
        "Fraud Cases Detected",
        "Total Fraud Cases",
        "Transactions Flagged"
    ]
].copy()

print("\n" + "=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

display(
    comparison_display.style.format({
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1-Score": "{:.4f}"
    })
)


# =============================================================================
# MODEL METRIC COMPARISON PLOT
# =============================================================================

metrics = [
    "Precision",
    "Recall",
    "F1-Score"
]

comparison_plot = (
    comparison_df
    .set_index("Model")[metrics]
)

comparison_plot.plot(
    kind="bar",
    figsize=(10, 6)
)

plt.title(
    "Autoencoder vs Isolation Forest vs One-Class SVM"
)

plt.xlabel("Model")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(title="Metric")

plt.tight_layout()
plt.show()


# =============================================================================
# FRAUD CASES DETECTED COMPARISON
# =============================================================================

fraud_detection_plot = comparison_df.set_index(
    "Model"
)["Fraud Cases Detected"]

fraud_detection_plot.plot(
    kind="bar",
    figsize=(9, 5)
)

plt.title(
    "Known Fraud Cases Detected by Each Model"
)

plt.xlabel("Model")
plt.ylabel("Number of Fraud Cases Detected")
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()


# =============================================================================
# RECONSTRUCTION ERROR: NORMAL VS FRAUD
# =============================================================================
#
# At this stage labels are used ONLY to visualize/evaluate the held-out test
# set. They were never used during Autoencoder training or threshold selection.
# =============================================================================

test_error_df = pd.DataFrame({
    "Reconstruction_Error": test_errors,
    "Actual_Class": y_test.values
})

test_error_df["Transaction_Type"] = (
    test_error_df["Actual_Class"]
    .map({
        0: "Normal",
        1: "Fraud"
    })
)

plt.figure(figsize=(10, 6))

sns.boxplot(
    data=test_error_df,
    x="Transaction_Type",
    y="Reconstruction_Error"
)

plt.axhline(
    THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=f"Threshold = {THRESHOLD:.6f}"
)

plt.title(
    "Reconstruction Error: Normal vs Fraud"
)

plt.xlabel("Actual Transaction Type")
plt.ylabel("Reconstruction Error")

plt.legend()

plt.tight_layout()
plt.show()


# =============================================================================
# CREATE STREAMLIT-READY TEST RESULTS
# =============================================================================

test_results = X_test.copy()

test_results["Actual_Class"] = y_test.values

test_results["Reconstruction_Error"] = (
    test_errors
)

test_results["Predicted_Anomaly"] = (
    ae_predictions
)

test_results["Detection_Status"] = np.where(
    ae_predictions == 1,
    "Anomaly",
    "Normal"
)

test_results["Reason"] = np.where(
    ae_predictions == 1,
    "Reconstruction error exceeded the anomaly threshold.",
    "Reconstruction error remained within the normal range."
)

test_results["Anomaly_Score"] = (
    test_errors
)

print("\nTest results created successfully.")

display(
    test_results.head()
)


# =============================================================================
# FLAGGED TRANSACTIONS
# =============================================================================

flagged_transactions = test_results[
    test_results["Predicted_Anomaly"] == 1
].copy()

flagged_transactions = (
    flagged_transactions
    .sort_values(
        by="Reconstruction_Error",
        ascending=False
    )
)

print("\n" + "=" * 80)
print("FLAGGED TRANSACTIONS")
print("=" * 80)

print(
    "\nTotal transactions flagged:",
    len(flagged_transactions)
)

display(
    flagged_transactions[
        [
            "Time",
            "Amount",
            "Reconstruction_Error",
            "Actual_Class",
            "Detection_Status",
            "Reason"
        ]
    ].head(20)
)


# =============================================================================
# CREATE MODEL FOLDERS
# =============================================================================

os.makedirs(
    "/content/models",
    exist_ok=True
)

os.makedirs(
    "/content/results",
    exist_ok=True
)

print("\n✓ Model and results directories created.")


# =============================================================================
# SAVE AUTOENCODER
# =============================================================================

AUTOENCODER_PATH = (
    "/content/models/autoencoder.keras"
)

autoencoder.save(
    AUTOENCODER_PATH
)

print(
    "Saved:",
    AUTOENCODER_PATH
)


# =============================================================================
# SAVE SCALER
# =============================================================================

SCALER_PATH = (
    "/content/models/scaler.pkl"
)

joblib.dump(
    scaler,
    SCALER_PATH
)

print(
    "Saved:",
    SCALER_PATH
)


# =============================================================================
# SAVE AUTOENCODER THRESHOLD
# =============================================================================

THRESHOLD_PATH = (
    "/content/models/threshold.pkl"
)

threshold_data = {
    "threshold": float(THRESHOLD),
    "percentile": THRESHOLD_PERCENTILE,
    "method": (
        "Percentile of reconstruction errors "
        "on normal validation transactions"
    )
}

joblib.dump(
    threshold_data,
    THRESHOLD_PATH
)

print(
    "Saved:",
    THRESHOLD_PATH
)


# =============================================================================
# SAVE ISOLATION FOREST
# =============================================================================

IF_PATH = (
    "/content/models/isolation_forest.pkl"
)

joblib.dump(
    isolation_forest,
    IF_PATH
)

print(
    "Saved:",
    IF_PATH
)


# =============================================================================
# SAVE ONE-CLASS SVM
# =============================================================================

OCSVM_PATH = (
    "/content/models/one_class_svm.pkl"
)

joblib.dump(
    one_class_svm,
    OCSVM_PATH
)

print(
    "Saved:",
    OCSVM_PATH
)


# =============================================================================
# SAVE BASELINE THRESHOLDS
# =============================================================================

BASELINE_THRESHOLDS_PATH = (
    "/content/models/baseline_thresholds.pkl"
)

baseline_thresholds = {
    "isolation_forest_threshold": float(
        IF_THRESHOLD
    ),
    "one_class_svm_threshold": float(
        OCSVM_THRESHOLD
    ),
    "percentile": THRESHOLD_PERCENTILE
}

joblib.dump(
    baseline_thresholds,
    BASELINE_THRESHOLDS_PATH
)

print(
    "Saved:",
    BASELINE_THRESHOLDS_PATH
)


# =============================================================================
# SAVE TEST PREDICTIONS
# =============================================================================

RESULTS_PATH = (
    "/content/results/test_predictions.csv"
)

test_results.to_csv(
    RESULTS_PATH,
    index=False
)

print(
    "Saved:",
    RESULTS_PATH
)


# =============================================================================
# SAVE MODEL COMPARISON
# =============================================================================

COMPARISON_PATH = (
    "/content/results/model_comparison.csv"
)

comparison_df.to_csv(
    COMPARISON_PATH,
    index=False
)

print(
    "Saved:",
    COMPARISON_PATH
)


# =============================================================================
# SAVE THRESHOLD ANALYSIS
# =============================================================================

THRESHOLD_RESULTS_PATH = (
    "/content/results/threshold_analysis.csv"
)

threshold_df.to_csv(
    THRESHOLD_RESULTS_PATH,
    index=False
)

print(
    "Saved:",
    THRESHOLD_RESULTS_PATH
)


# =============================================================================
# SAVE PROJECT CONFIGURATION
# =============================================================================

CONFIG_PATH = (
    "/content/models/project_config.pkl"
)

project_config = {
    "random_seed": SEED,
    "features": FEATURES,
    "target": TARGET,
    "input_dimension": INPUT_DIM,
    "autoencoder_architecture": [
        INPUT_DIM,
        128,
        64,
        32,
        16,
        32,
        64,
        128,
        INPUT_DIM
    ],
    "threshold_percentile": THRESHOLD_PERCENTILE,
    "autoencoder_threshold": float(THRESHOLD),
    "isolation_forest_threshold": float(
        IF_THRESHOLD
    ),
    "one_class_svm_threshold": float(
        OCSVM_THRESHOLD
    )
}

joblib.dump(
    project_config,
    CONFIG_PATH
)

print(
    "Saved:",
    CONFIG_PATH
)


# =============================================================================
# FINAL PROJECT SUMMARY
# =============================================================================

print("\n")
print("=" * 80)
print("FINAL PROJECT SUMMARY")
print("=" * 80)

print("\nDATASET")
print("-" * 80)
print("Total transactions:", f"{len(df):,}")
print("Normal transactions:", f"{normal_count:,}")
print("Fraud transactions:", f"{fraud_count:,}")
print("Number of input features:", INPUT_DIM)

print("\nDATA SPLIT")
print("-" * 80)
print(
    "Normal training samples:",
    f"{len(X_train_normal):,}"
)
print(
    "Normal validation samples:",
    f"{len(X_val_normal):,}"
)
print(
    "Held-out test samples:",
    f"{len(X_test):,}"
)
print(
    "Fraud cases in held-out test:",
    int(y_test.sum())
)

print("\nAUTOENCODER")
print("-" * 80)
print(
    "Threshold percentile:",
    f"{THRESHOLD_PERCENTILE}%"
)
print(
    "Threshold:",
    f"{THRESHOLD:.8f}"
)
print(
    "Precision:",
    f"{ae_result['Precision']:.4f}"
)
print(
    "Recall:",
    f"{ae_result['Recall']:.4f}"
)
print(
    "F1-Score:",
    f"{ae_result['F1-Score']:.4f}"
)
print(
    "Fraud cases detected:",
    ae_result["Fraud Cases Detected"]
)
print(
    "Transactions flagged:",
    ae_result["Transactions Flagged"]
)
print(
    "ROC-AUC:",
    f"{ae_auc:.4f}"
)

print("\nISOLATION FOREST")
print("-" * 80)
print(
    "Precision:",
    f"{if_result['Precision']:.4f}"
)
print(
    "Recall:",
    f"{if_result['Recall']:.4f}"
)
print(
    "F1-Score:",
    f"{if_result['F1-Score']:.4f}"
)
print(
    "Fraud cases detected:",
    if_result["Fraud Cases Detected"]
)

print("\nONE-CLASS SVM")
print("-" * 80)
print(
    "Precision:",
    f"{ocsvm_result['Precision']:.4f}"
)
print(
    "Recall:",
    f"{ocsvm_result['Recall']:.4f}"
)
print(
    "F1-Score:",
    f"{ocsvm_result['F1-Score']:.4f}"
)
print(
    "Fraud cases detected:",
    ocsvm_result["Fraud Cases Detected"]
)


# =============================================================================
# LIST SAVED FILES
# =============================================================================

print("\n")
print("=" * 80)
print("SAVED PROJECT FILES")
print("=" * 80)

for root, dirs, files_list in os.walk(
    "/content/models"
):
    for file in files_list:
        print(
            os.path.join(
                root,
                file
            )
        )

for root, dirs, files_list in os.walk(
    "/content/results"
):
    for file in files_list:
        print(
            os.path.join(
                root,
                file
            )
        )


# =============================================================================
# CREATE ZIP OF ALL TRAINED ARTIFACTS
# =============================================================================
#
# This makes it easier to download all trained files at once.
# =============================================================================

import shutil

ZIP_BASE = "/content/fraud_detection_models"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    "/content",
    "models"
)

print(
    "\n✓ All model artifacts compressed."
)

print(
    "ZIP file:",
    ZIP_BASE + ".zip"
)